In [0]:
import os

project_root = os.path.dirname(os.getcwd())
print("Project root:", project_root)

%pip install -e $project_root
dbutils.library.restartPython()


In [0]:
import os
print(os.listdir(os.path.dirname(os.getcwd())))

In [0]:
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
print("PROJECT_ROOT =", PROJECT_ROOT)


In [0]:
import movielens
print("OK:", movielens.__file__)

In [0]:
%pip install -e ..
dbutils.library.restartPython()

In [0]:
# Databricks notebook source
from __future__ import annotations

# COMMAND ----------

# 0) Imports + config
import os
import sys
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql import types as T

# COMMAND ----------

# 1) Project import fix (NO pip editable install)
# -------------------------------------------------------------------
# We add your repo /src to PYTHONPATH so: `import movielens...` works.
# Update REPO_ROOT if needed.
REPO_ROOT = "/Workspace/Users/pierrick.vanhoecke@bunkco.be"  # <-- change if your repo is elsewhere
SRC_PATH = f"{REPO_ROOT}/src"

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print("sys.path[0] =", sys.path[0])

# Optional debug:
# nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
# print("notebookPath =", nb_path)

# COMMAND ----------

# 2) Read TMDB token from Databricks Secrets
# -------------------------------------------------------------------
# Scope: movielens
# Key:   tmdb_read_token
TMDB_READ_TOKEN = dbutils.secrets.get("movielens", "tmdb_read_token")
if not TMDB_READ_TOKEN or not TMDB_READ_TOKEN.strip():
    raise ValueError("Secret movielens/tmdb_read_token is empty. Set it and re-run.")

os.environ["TMDB_READ_TOKEN"] = TMDB_READ_TOKEN.strip()
print("TMDB token loaded from secrets (not printing token)")

# COMMAND ----------

# 3) Paths (your existing Volume)
# -------------------------------------------------------------------
BASE_VOL = "dbfs:/Volumes/workspace/movielens/movielens_files"
OUT_DIR = f"{BASE_VOL}/scraped"
OUT_JSONL = f"{OUT_DIR}/scraped_metadata.jsonl"

print("BASE_VOL :", BASE_VOL)
print("OUT_DIR  :", OUT_DIR)
print("OUT_JSONL:", OUT_JSONL)

dbutils.fs.mkdirs(OUT_DIR)

# remove previous jsonl for idempotency
try:
    dbutils.fs.rm(OUT_JSONL, recurse=True)
    print("Removed existing:", OUT_JSONL)
except Exception as e:
    print("Nothing to remove (ok):", str(e)[:200])

# COMMAND ----------

# 4) Load candidate movies to scrape (from Gold dim)
# -------------------------------------------------------------------
# We use your existing UC schema naming.
GOLD = "workspace.movielens_gold"
movies = (
    spark.table(f"{GOLD}.dim_movies_enriched")
    .select("movieId", "tmdbId", "title")
    .where(F.col("tmdbId").isNotNull())
)

n_movies = movies.count()
print("Movies with tmdbId:", n_movies)
display(movies.limit(10))

# COMMAND ----------

# 5) Scrape TMDB and write JSONL
# -------------------------------------------------------------------

try:
    from movielens.scrape_tmdb import main as scrape_main
except Exception as e:
    raise ImportError(
        "Cannot import 'movielens'.\n"
        "Make sure your repo has /src/movielens and REPO_ROOT/SRC_PATH is correct.\n"
        f"Original error: {e}"
    )

sample_limit = 500 
cand = movies.orderBy(F.col("movieId").asc()).limit(sample_limit).collect()
items = [{"movieId": int(r["movieId"]), "tmdbId": str(r["tmdbId"])} for r in cand]

print(f"Scraping {len(items)} items…")
scrape_main(items=items, output_path=OUT_JSONL)

print("Wrote JSONL to:", OUT_JSONL)

# COMMAND ----------

# 6) Read JSONL into Spark and write Bronze table
# -------------------------------------------------------------------
BRONZE = "workspace.movielens_bronze"
BRONZE_TABLE = f"{BRONZE}.bronze_scraped_metadata"

schema = T.StructType(
    [
        T.StructField("movieId", T.LongType(), True),
        T.StructField("tmdbId", T.StringType(), True),
        T.StructField("director", T.StringType(), True),
        T.StructField("budget", T.LongType(), True),
        T.StructField("poster_url", T.StringType(), True),
        T.StructField("source", T.StringType(), True),
        T.StructField("ingested_at", T.StringType(), True),
    ]
)

meta = spark.read.schema(schema).json(OUT_JSONL)

# Normalize & add ingestion timestamp
meta = (
    meta.withColumn("movieId", F.col("movieId").cast("long"))
        .withColumn("budget", F.col("budget").cast("long"))
        .withColumn("ingested_at", F.col("ingested_at").cast("string"))
        .withColumn("ingested_at_ts", F.to_timestamp("ingested_at"))
        .withColumn("ingested_date", F.to_date("ingested_at_ts"))
)

print("BRONZE_TABLE:", BRONZE_TABLE)
print("Rows:", meta.count())
display(meta.limit(20))

(
    meta.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

print("Wrote Bronze table:", BRONZE_TABLE, "rows=", spark.table(BRONZE_TABLE).count())

# COMMAND ----------

# 7) Quick checks
# -------------------------------------------------------------------
# a) nulls
display(
    spark.table(BRONZE_TABLE)
    .select(
        F.count("*").alias("n_rows"),
        F.sum(F.col("movieId").isNull().cast("int")).alias("null_movieId"),
        F.sum(F.col("tmdbId").isNull().cast("int")).alias("null_tmdbId"),
        F.sum(F.col("director").isNull().cast("int")).alias("null_director"),
        F.sum(F.col("poster_url").isNull().cast("int")).alias("null_poster_url"),
    )
)

# b) sample
display(spark.table(BRONZE_TABLE).orderBy(F.col("movieId").asc()).limit(30))
